# EDA: Tìm Hot Partition trong hệ thống chat cũ

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.datastax.spark:spark-cassandra-connector_2.12:3.4.1 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, min, max

spark = SparkSession.builder \
    .appName("EDA_HotPartition_Analysis") \
    .config("spark.cassandra.connection.host", os.getenv("CASSANDRA_HOST", "cassandra-source")) \
    .config("spark.cassandra.connection.port", os.getenv("CASSANDRA_PORT", "9042")) \
    .getOrCreate()

print("ok")

In [ ]:
df = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="chat_table", keyspace="chat_system") \
    .load()

print(f"Total: {df.count()}")

In [ ]:
df.printSchema()

for c in df.columns:
    n = df.filter(col(c).isNull()).count()
    if n > 0:
        print(f"{c}: {n} nulls")

In [ ]:
# range thời gian
ts = df.select(min('timestamp').alias('min_ts'), max('timestamp').alias('max_ts')).collect()[0]
print(f"From {ts['min_ts']} -> {ts['max_ts']}")

In [ ]:
room_counts = df.groupBy("room_id") \
    .agg(count("message_id").alias("cnt")) \
    .orderBy(col("cnt").desc()) \
    .limit(10).toPandas()

room_counts.head(10)

In [ ]:
top10 = room_counts.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top10["room_id"], top10["cnt"], color='salmon')
plt.title("Top 10 Room theo số lượng tin nhắn")
plt.xlabel("Room ID")
plt.ylabel("Messages")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**=> room_999 chiếm ~80% data, rõ ràng là Hot Partition. Schema cũ chỉ dùng `room_id` làm PK nên toàn bộ msg dồn vào 1 partition duy nhất.**